### SILVER Transformation of bronze.payments

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

### Step1: Read bronz.payments

In [0]:
df = spark.table("bronze.payments").select("*")

In [0]:
df.count()

In [0]:
df.limit(5).display()

In [0]:
df.select(countDistinct(col("order_id")).alias("count")).display()

In [0]:
df.createTempView("payments")

In [0]:
df = spark.sql("""
        SELECT 
            *,
            ROUND(SUM(payment_value) OVER (PARTITION BY order_id), 2) as total_payment
        FROM payments
""")


In [0]:

df = df.withColumn("payment_value", round(sum(col("payment_value"))\
            .over(Window.partitionBy(col("order_id"))),2))


In [0]:
df.display()

In [0]:
df.count()

In [0]:
w= Window.partitionBy(col("order_id")).orderBy(col("_load_timestamp").desc())
df = df.withColumn("rn", row_number().over(w)).filter(col("rn")==1).drop("rn").drop("total_payment")
df.count()

### Null Check

In [0]:
df = df.filter(col("order_id").isNotNull())

### write to silver

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

In [0]:
df.write.format("delta").mode("overwrite").option("inferSchema", True).saveAsTable("silver.payments")

In [0]:
df_silver = spark.table("silver.payments")
df_silver.limit(5).display()